In [0]:
%sql

SELECT COUNT(*) FROM
bootcamp.silver.propiedades;
DESCRIBE TABLE bootcamp.silver.propiedades;

In [0]:
%sql
use bootcamp.silver;

select count(*),
 count(distinct (estado)) as estado,
count(distinct (partido)) as partido,
count(distinct (region)) as region,
count(distinct (tipo_operacion)) as tipo_operacion,
count(distinct (precio)) as precio,
count(distinct (moneda)) as moneda,
count(distinct (ambientes)) as ambientes,
count(distinct (cochera)) as cochera,
count(distinct (orientacion)) as orientacion,
count(distinct (url)) as url

 from bootcamp.silver.propiedades

In [0]:
%sql

CREATE or REPLACE TEMP VIEW stagin_correcciones AS
    SELECT partido,region,'Gran Buenos Aires' AS ciudad
    from bootcamp.gold.dim_zona
    WHERE region ='gba zona norte' and ciudad = 'GBA';

select * from stagin_correcciones


In [0]:
%sql

SELECT COUNT(*) AS count_antes FROM bootcamp.gold.dim_zona;


In [0]:
%sql

merge into bootcamp.gold.dim_zona AS Target
USING stagin_correcciones as SOURCE
ON target.partido=source.partido and target.region=source.region
when matched then
UPDATE SET ciudad = source.ciudad; 

In [0]:
%sql

drop table bootcamp.gold.dim_zona_scd2

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2  (
zona_id BIGINT GENERATED ALWAYS AS IDENTITY,
partido string not NULL,
region STRING not null,
ciudad STRING,
provincia string DEFAULT "Buenos Aires",
pais STRING default "Argentina",
valid_from TIMESTAMP not null default current_timestamp(),
valid_to TIMESTAMP default "9999-12-31",
is_current boolean default true
) 

USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
;



In [0]:
%sql

INSERT INTO bootcamp.gold.dim_zona_scd2 (
    partido,
    region,
    ciudad
)
SELECT distinct partido, region,
case when region = 'capital federal' then 'CABA'
else 'GBA' end as ciudad
from bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql

SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

In [0]:
%sql

SELECT * from bootcamp.gold.dim_zona_scd2

In [0]:
%sql

update  bootcamp.gold.dim_zona_scd2
set valid_to = current_timestamp(), is_current=false
where partido = 'capital federal' and ciudad = 'CABA' and is_current=true


In [0]:
%sql
-- Paso 3: Verificar que todos son is_current = TRUE
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

In [0]:
%sql

update  bootcamp.gold.dim_zona_scd2
set valid_to = current_timestamp(), is_current=false
where partido = 'capital federal' and ciudad = 'CABA' and is_current=true

In [0]:
%sql
insert into bootcamp.gold.dim_zona_scd2 (
    partido,region,ciudad,
    valid_from,valid_to,is_current
)
values (
    'capital federal','capital federal','Buenos Aires',
    current_timestamp(),'9999-12-31',TRUE
)

In [0]:
%sql

SELECT * FROM
bootcamp.gold.dim_zona_scd2 WHERE partido = 'capital federal'
ORDER BY valid_from

In [0]:
%sql

create or replace temp view staging_cambios_scd2  as
select *  from VALUES
('palermo', 'capital federal', 'CABA Sur'),
  ('barrio-nuevo', 'capital federal', 'CABA')
AS t(partido, region, ciudad_nueva)


In [0]:
%sql

MERGE into bootcamp.gold.dim_zona_scd2 TARGET
USING staging_cambios_scd2 SOURCE
ON target.partido = source.partido
   AND target.region = source.region
   AND target.is_current = TRUE
WHEN MATCHED AND target.ciudad != source.ciudad_nueva THEN
  UPDATE SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad, valid_from, valid_to, is_current)
  VALUES (source.partido, source.region, source.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: INSERT nuevas versiones para los que se cerraron
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
SELECT s.partido, s.region, s.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE
FROM staging_cambios_scd2 s
INNER JOIN bootcamp.gold.dim_zona_scd2 t
  ON s.partido = t.partido AND s.region = t.region
  AND t.is_current = FALSE AND t.valid_to >= CURRENT_TIMESTAMP() - INTERVAL 1 MINUTE
WHERE NOT EXISTS (
  SELECT 1 FROM bootcamp.gold.dim_zona_scd2 t2
  WHERE t2.partido = s.partido AND t2.region = s.region AND t2.is_current = TRUE
);

In [0]:
%sql
-- Paso 4: Verificar - palermo debe tener 2 versiones, barrio-nuevo 1
SELECT partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido IN ('palermo', 'barrio-nuevo')
ORDER BY partido, valid_from;

In [0]:
%sql

CREATE TABLE IF NOT EXISTs bootcamp.gold.dim_zona_scd3 (

    zona_id bigint GENERATED ALWAYS AS IDENTITY,
    partido string not null,
    region string not null,
    ciudad string,
    ciudad_anterior string,
    fecha_ultima_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    provincia        STRING DEFAULT 'Buenos Aires',
  pais             STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (zona_id)
   
) USING DELTA
COMMENT 'Dimension de zonas -- SCD Type 3 (valor actual + anterior)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql

INSERT INTO bootcamp.gold.dim_zona_scd3 (
    partido,region,ciudad)
    SELECT distinct partido, region,
    CASE WHEN region = "capital federal" then "CABA"
    ELSE "GBA" end
from bootcamp.silver.propiedades
where partido is not null

In [0]:
%sql
select * from bootcamp.gold.dim_zona_scd3

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW source_cambios AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('belgrano', 'capital federal', 'CABA Norte')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql

merge into bootcamp.gold.dim_zona_scd3 AS target
USING source_cambios as source
on target.partido=source.partido 
and target.region=source.region
WHEN matched 
AND target.ciudad!=source.ciudad_nueva
THEN update set
ciudad_anterior=target.ciudad,
ciudad=source.ciudad_nueva,
fecha_ultima_actualizacion=current_timestamp()
WHEN not matched 
THEN insert(partido,region,ciudad)
VALUES(partido,region,ciudad_nueva)

In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3 


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW source_cambios_2 AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA SurOeste'),
  ('belgrano', 'capital federal', 'CABA NorEste')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql

merge into bootcamp.gold.dim_zona_scd3 AS target
USING source_cambios_2 as source
on target.partido=source.partido 
and target.region=source.region
WHEN matched 
AND target.ciudad!=source.ciudad_nueva
THEN update set
ciudad_anterior=target.ciudad,
ciudad=source.ciudad_nueva,
fecha_ultima_actualizacion=current_timestamp()
WHEN not matched 
THEN insert(partido,region,ciudad)
VALUES(partido,region,ciudad_nueva)

In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3
where ciudad_anterior is not null 